In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install -q ultralytics==8.4.149 onnx onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 96.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 86.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 9.1 MB/s eta 0:00:00


In [3]:
from pathlib import Path

import cv2
import json
import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import Dataset, DataLoader
from zipfile import ZipFile

import copy
import shutil
import statistics
import time

import onnx
import onnxruntime as ort

import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [4]:
DATASET_URL = (
    "https://github.com/ultralytics/"
    "assets/releases/download/v0.0.0/"
    "crack-seg.zip"
)

DATASETS_ROOT = Path("/content/datasets")
ARCHIVE_PATH = DATASETS_ROOT / "crack-seg.zip"
DATASET_ROOT = DATASETS_ROOT / "crack-seg"

DATASETS_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = ("train", "val", "test")
EXPECTED_COUNTS = {"train": 3717, "val": 200, "test": 112}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

dataset_ready = (DATASET_ROOT / "images" / "train").is_dir()

if not dataset_ready:
    if not ARCHIVE_PATH.is_file():
        torch.hub.download_url_to_file(
            DATASET_URL, str(ARCHIVE_PATH), progress=True
        )

    with ZipFile(ARCHIVE_PATH, "r") as zip_file:
        zip_file.extractall(DATASETS_ROOT)


if not DATASET_ROOT.is_dir():
    candidates = [
        path for path in DATASETS_ROOT.rglob("*")
        if path.is_dir() and (path / "images").is_dir() and "crack" in path.name.lower()
    ]

    if len(candidates) != 1:
        candidates = [
            path for path in DATASETS_ROOT.rglob("images")
            if path.is_dir()
        ]
        if len(candidates) == 1:
            candidates = [candidates[0].parent]


    DATASET_ROOT = candidates[0]

for split in SPLITS:
    required_directories = [
        (DATASET_ROOT / "images" / split),
        (DATASET_ROOT / "labels" / split)
    ]
    
    for directory in (required_directories):
        if not directory.is_dir():
            raise FileExistsError(directory)

print("Dataset extracted:", DATASET_ROOT)

100%|██████████| 91.6M/91.6M [00:00<00:00, 109MB/s] 


Dataset extracted: /content/datasets


In [5]:
DRIVE_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs"
)


DATA_YAML_PATH = (
    DRIVE_OUTPUT_ROOT
    / "block_02" / "crack_seg_local.yaml"
)

TEST_IMAGES_DIR = DATASET_ROOT / "images" / "test"
TEST_LABELS_DIR = DATASET_ROOT / "labels" / "test"

UNET_CHECKPOINT = (
    DRIVE_OUTPUT_ROOT
    / "block_03"
    / "v1_bce_dice"
    / "best.pt"
)

YOLO_CHECKPOINT = (
    DRIVE_OUTPUT_ROOT
    / "block_04"
    / "yolo26n_crack_seg_v1"
    / "weights"
    / "best.pt"
)

OUTPUT_ROOT = DRIVE_OUTPUT_ROOT / "block_06"
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)

**Lock all decisions**

In [6]:
UNET_IMAGE_SIZE = 416
UNET_THRESHOLD = 0.45

YOLO_IMAGE_SIZE = 640
YOLO_AP_CONFIDENCE = 0.001
YOLO_NMS_IOU = 0.70

TEST_SPLIT_USED_FOR_TUNING = False

locked_protocol = {
    "unet_model": "resnet34_unet_bce_dice",
    "unet_image_size": UNET_IMAGE_SIZE,
    "unet_threshold": UNET_THRESHOLD,
    "yolo_checkpoint": str(YOLO_CHECKPOINT),
    "yolo_image_size": YOLO_IMAGE_SIZE,
    "yolo_ap_confidence": YOLO_AP_CONFIDENCE,
    "yolo_nms_iou": YOLO_NMS_IOU,
    "test_split_used_for_tuning": False,
}

protocol_path = OUTPUT_ROOT / "evaluation_protocol.json"

protocol_path.write_text(
    json.dumps(locked_protocol, indent=2), encoding="utf-8"
)

print(json.dumps(locked_protocol, indent=2))

{
  "unet_model": "resnet34_unet_bce_dice",
  "unet_image_size": 416,
  "unet_threshold": 0.45,
  "yolo_checkpoint": "/content/drive/MyDrive/vision_unit_02_outputs/block_04/yolo26n_crack_seg_v1/weights/best.pt",
  "yolo_image_size": 640,
  "yolo_ap_confidence": 0.001,
  "yolo_nms_iou": 0.7,
  "test_split_used_for_tuning": false
}


## U-Net semantic test evaluation

**Polygon → merged semantic mask**


In [7]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp",}


def load_semantic_mask(label_path, image_height, image_width,):
    mask = np.zeros(
        (image_height, image_width,),
        dtype=np.uint8
    )

    if not label_path.is_file():
        return mask

    lines = label_path.read_text(encoding="utf-8",).splitlines()

    for line in lines:
        values = line.strip().split()

        if len(values) < 7:
            continue

        coordinates = np.asarray(
            values[1:], dtype=np.float32
        ).reshape(-1, 2)

        if len(coordinates) < 3:
            continue

        coordinates = np.clip(
            coordinates,
            0.0, 1.0
        )

        coordinates[:, 0] *= image_width
        coordinates[:, 1] *= image_height

        coordinates[:, 0] = np.clip(
            coordinates[:, 0],
            0, image_width - 1
        )

        coordinates[:, 1] = np.clip(
            coordinates[:, 1],
            0, image_height - 1
        )

        polygon = np.rint(coordinates).astype(np.int32)

        cv2.fillPoly(
            mask,
            [polygon],
            color=1,
        )

    return mask

**Test dataset**

In [8]:
IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)

IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)

In [9]:
class CrackSemanticTestDataset(Dataset):
    def __init__(self, images_dir, labels_dir, image_size):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        
        self.image_size = image_size
        self.images_paths = sorted(
            path for path in self.images_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
    
    def __len__(self):
        return len(self.images_paths)

    def __getitem__(self, index):
        image_path = self.images_paths[index]
        label_path = (
            self.labels_dir
            / f"{image_path.stem}.txt"
        )
        
        image_bgr = cv2.imread(str(image_path))
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        height, width = (image_rgb.shape[:2])
    
        mask = load_semantic_mask(label_path, height, width)
        
        image_rgb = cv2.resize(
            image_rgb,
            (self.image_size,self.image_size),
            interpolation=cv2.INTER_LINEAR
        )
        
        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        )

        image_float = image_rgb.astype( np.float32) / 255.0
        image_float = (image_float - IMAGENET_MEAN
        ) / IMAGENET_STD

        image_tensor = torch.from_numpy(image_float).permute(2, 0, 1).float()
        
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "image_id": image_path.stem,
        }

In [10]:
test_dataset = (
    CrackSemanticTestDataset(
        TEST_IMAGES_DIR,
        TEST_LABELS_DIR,
        UNET_IMAGE_SIZE,
    )
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print( "Test images:",len(test_dataset))

Test images: 112


**Load selected U-Net**

In [11]:
class DoubleConv(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        
        self.block = nn.Sequential(
            nn.Conv2d(
                input_channels, output_channels,
                kernel_size=3, padding=1, bias=False
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(output_channels, output_channels,
                      kernel_size=3, padding=1, bias=False
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)

class DecoderBlock(nn.Module):
    def __init__(self, input_channels, skip_channels, output_channels):
        super().__init__()
        
        self.convolution = DoubleConv(input_channels + skip_channels, output_channels)
    
    def forward(self, inputs, skip):
        inputs = F.interpolate(
            inputs, size=skip.shape[-2:],
            mode="bilinear", align_corners=False
        )
        combined = torch.cat([inputs, skip], dim=1)
        return self.convolution(combined)
    

In [12]:
class ResNet34UNet(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        
        backbone = resnet34(weights=weights)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu
        )
        self.pool = backbone.maxpool
        
        self.encoder1 = backbone.layer1
        self.encoder2 = backbone.layer2
        self.encoder3 = backbone.layer3
        self.encoder4 = backbone.layer4
        
        self.decoder_4 = DecoderBlock(
            input_channels=512, skip_channels=256, output_channels=256
        )
        self.decoder_3 = DecoderBlock(
            input_channels=256, skip_channels=128, output_channels=128
        )
        self.decoder_2 = DecoderBlock(
            input_channels=128, skip_channels=64, output_channels=64
        )
        self.decoder_1 = DecoderBlock(
            input_channels=64, skip_channels=64, output_channels=32
        )
        
        self.final_refinement = DoubleConv(
            input_channels=32, output_channels=32
        )
        
        self.segmentation_head = nn.Conv2d(32, 1, kernel_size=1)


    def forward(self, inputs):
        input_size = inputs.shape[-2:]
        
        skip_0 = self.stem(inputs)
        skip_1 = self.encoder1(self.pool(skip_0))
        skip_2 = self.encoder2(skip_1)
        skip_3 = self.encoder3(skip_2)
        
        bottleneck = self.encoder4(skip_3)
        
        decoded = self.decoder_4(bottleneck, skip_3)
        decoded = self.decoder_3(decoded, skip_2)
        decoded = self.decoder_2(decoded, skip_1)
        decoded = self.decoder_1(decoded, skip_0)
        
        decoded = F.interpolate(
            decoded, size=input_size,
            mode="bilinear", align_corners=False
        )
        
        decoded = self.final_refinement(decoded)
        logits = self.segmentation_head(decoded)
        
        return logits

In [13]:
device = torch.device("cuda"if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    UNET_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

state_dict = checkpoint["model_state"]
state_dict = {
    key.removeprefix("module."): value
    for key, value in state_dict.items()
}

unet_model = ResNet34UNet(weights=None)
unet_model.load_state_dict(state_dict, strict=True)

unet_model = unet_model.to(device).eval()
parameter_count = sum(
    parameter.numel()
    for parameter in unet_model.parameters()
)

print("Device:", device)
print("Parameters:", f"{parameter_count:,}")
print("Training epoch:", checkpoint.get("epoch"))
print("Validation metrics:",checkpoint.get("validation_metrics"))
print("Experiment:", checkpoint.get("experiment"))

Device: cuda
Parameters: 24,447,841
Training epoch: 2
Validation metrics: {'tp': 540357, 'fp': 148880, 'fn': 161041, 'tn': 33760922, 'precision': 0.7839930241702056, 'recall': 0.7703999726260982, 'iou': 0.6355062697141406, 'dice': 0.7771370632840393, 'pixel_accuracy': 0.9910456441845414, 'loss': 0.2765188832581043}
Experiment: bce_dice


**Semantic metrics**

In [14]:
def safe_divide(numerator, denominator, empty_value=0.0):
    if denominator == 0:
        return empty_value

    return (numerator / denominator)

def metrics_from_counts(tp, fp, fn, tn,):
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    
    iou = safe_divide(tp, tp + fp + fn)
    dice = safe_divide(2 * tp, 2 * tp + fp + fn)

    pixel_accuracy = safe_divide(tp + tn, tp + fp + fn + tn)
    
    return {
        "precision": precision,
        "recall": recall,
        "iou": iou,
        "dice": dice,
        "pixel_accuracy": pixel_accuracy
    }

**Run final U-Net test once**

In [15]:
aggregate = { "tp": 0, "fp": 0, "fn": 0, "tn": 0}
per_image_rows = []

with torch.inference_mode():
    for batch in test_loader:
        images = batch["image"].to(device)
        targets = batch["mask"].to(device)

        logits = unet_model(images)
        
        probabilities = torch.sigmoid(logits)
        predictions = probabilities >= UNET_THRESHOLD

        targets_bool = targets >= 0.5
        
        for index, image_id in enumerate(batch["image_id"]):
            prediction = predictions[index]
            target = targets_bool[index]

            tp = (prediction & target).sum().item()
            fp = (prediction & ~target).sum().item()

            fn = (~prediction & target).sum().item()
            tn = (~prediction & ~target).sum().item()
            
            for key, value in {
                "tp": tp, "fp": fp,
                "fn": fn, "tn": tn,
            }.items():
                aggregate[key] += value

            row_metrics = metrics_from_counts(tp, fp, fn, tn)
            per_image_rows.append({
                "image_id": image_id,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "tn": tn,
                "gt_foreground_pixels": (tp + fn),
                "predicted_foreground_pixels": (tp + fp),
                **row_metrics
            })

In [16]:
semantic_metrics = metrics_from_counts(**aggregate)

per_image_dataframe = pd.DataFrame(per_image_rows)
per_image_dataframe.head()

,image_id,tp,fp,fn,tn,gt_foreground_pixels,predicted_foreground_pixels,precision,recall,iou,dice,pixel_accuracy
0,1616.rf.c868709931a671796794fdbb95352c5a,2665,1429,163,168799,2828,4094,0.650953,0.942362,0.626028,0.770009,0.990801
1,1675.rf.e3aa3f8d28d0247ef0284dd46dacc29f,2485,1118,1720,167733,4205,3603,0.689703,0.590963,0.466842,0.636527,0.983601
2,1686.rf.809fb1b51c607e5cf787e44ef4ddd7b8,5366,845,2865,163980,8231,6211,0.863951,0.651926,0.591230,0.743110,0.978562
3,1706.rf.011d213c21ec78896c36728dcbc156f5,3438,1580,4849,163189,8287,5018,0.685134,0.414867,0.348434,0.516798,0.962850
4,1716.rf.85ea38b36008beaa72c5d8541f734eb0,3442,822,963,167829,4405,4264,0.807223,0.781385,0.658504,0.794094,0.989685


In [17]:
semantic_report = {
    "model": "U-Net ResNet34 BCE+Dice",
    "test_images": len(test_dataset),
    "image_size": UNET_IMAGE_SIZE,
    "threshold": UNET_THRESHOLD,
    **aggregate,
    **semantic_metrics,
    "mean_per_image_dice": float(
        per_image_dataframe["dice"].mean()
    ),
    "mean_per_image_iou": float(
        per_image_dataframe["iou"].mean()
    ),
    "test_split_used_for_tuning": False,
}

semantic_metrics_path = OUTPUT_ROOT / "semantic_test_metrics.json"
semantic_per_image_path = OUTPUT_ROOT / "semantic_per_image_metrics.csv"


semantic_metrics_path.write_text(
    json.dumps(semantic_report,indent=2), encoding="utf-8"
)

per_image_dataframe.to_csv(semantic_per_image_path,index=False)
print(json.dumps(semantic_report,indent=2))

{
  "model": "U-Net ResNet34 BCE+Dice",
  "test_images": 112,
  "image_size": 416,
  "threshold": 0.45,
  "tp": 294745,
  "fp": 93396,
  "fn": 89197,
  "tn": 18904934,
  "precision": 0.7593761030141108,
  "recall": 0.7676810559928322,
  "iou": 0.6174765051179667,
  "dice": 0.7635059961169978,
  "pixel_accuracy": 0.9905793809931055,
  "mean_per_image_dice": 0.7534098759669722,
  "mean_per_image_iou": 0.6226315141881887,
  "test_split_used_for_tuning": false
}


## YOLO26-seg instance test evaluation

In [18]:
yolo_model = YOLO(str(YOLO_CHECKPOINT))

yolo_test_metrics = (
    yolo_model.val(
        data=str(DATA_YAML_PATH),
        split="test",
        imgsz=YOLO_IMAGE_SIZE,
        batch=16,
        conf=YOLO_AP_CONFIDENCE,
        iou=YOLO_NMS_IOU,
        device=0,
        rect=False,
        plots=True,
        project=str(OUTPUT_ROOT),
        name="yolo_pytorch_test",
        exist_ok=True,
        verbose=True,
    )
)

Ultralytics 8.4.149 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-seg summary (fused): 136 layers, 2,689,079 parameters, 0 gradients, 9.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 903.3±579.3 MB/s, size: 35.0 KB)
val: Scanning /content/datasets/labels/test... 112 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 112/112 666.4it/s 0.2s5s
val: New cache created: /content/datasets/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.4it/s 4.9s0.5ss
                   all        112        148      0.758      0.634      0.712      0.415       0.65      0.541      0.546      0.179
Speed: 6.8ms preprocess, 6.1ms inference, 0.0ms loss, 4.7ms postprocess per image
Results saved to /content/drive/MyDrive/vision_unit_02_outputs/block_06/yolo_pytorch_test


In [19]:
instance_report = {
    "model": "YOLO26n-seg",
    "test_images": 112,
    "image_size": YOLO_IMAGE_SIZE,
    "ap_collection_confidence": YOLO_AP_CONFIDENCE,
    "nms_iou": YOLO_NMS_IOU,
    "box_precision": float(yolo_test_metrics.box.mp),
    "box_recall": float(yolo_test_metrics.box.mr),
    "box_map50": float(yolo_test_metrics.box.map50),
    "box_map75": float(yolo_test_metrics.box.map75),
    "box_map50_95": float(yolo_test_metrics.box.map),
    "mask_precision": float(yolo_test_metrics.seg.mp),
    "mask_recall": float(yolo_test_metrics.seg.mr),
    "mask_map50": float(yolo_test_metrics.seg.map50),
    "mask_map75": float(yolo_test_metrics.seg.map75),
    "mask_map50_95": float(yolo_test_metrics.seg.map),
    "speed_ms": {
        key: float(value)
        for key, value in yolo_test_metrics.speed.items()
    },
    "test_split_used_for_tuning": False,
}

instance_metrics_path = OUTPUT_ROOT / "instance_test_metrics.json"

instance_metrics_path.write_text(
    json.dumps(instance_report, indent=2),
    encoding="utf-8"
)

print(json.dumps(instance_report,indent=2))

{
  "model": "YOLO26n-seg",
  "test_images": 112,
  "image_size": 640,
  "ap_collection_confidence": 0.001,
  "nms_iou": 0.7,
  "box_precision": 0.7576986670287778,
  "box_recall": 0.6338783940366932,
  "box_map50": 0.7123491510412542,
  "box_map75": 0.4656606864104382,
  "box_map50_95": 0.41475297723290955,
  "mask_precision": 0.6504312178840683,
  "mask_recall": 0.5405997698252373,
  "mask_map50": 0.5464061625500753,
  "mask_map75": 0.04065054503146418,
  "mask_map50_95": 0.17943502915224563,
  "speed_ms": {
    "preprocess": 6.764344812499497,
    "inference": 6.149075214285712,
    "loss": 0.01249771428543259,
    "postprocess": 4.653485187499688
  },
  "test_split_used_for_tuning": false
}


In [20]:
completion_flag_path = OUTPUT_ROOT / "stage_6a_complete.txt"

completion_flag_path.write_text(
    "Held-out test evaluated with locked configuration.\n",
    encoding="utf-8",
)

print("=" * 60)
print("BLOCK 6A COMPLETE")
print("=" * 60)

print("U-Net test Dice:", round(semantic_report["dice"], 4))
print("U-Net test IoU:", round(semantic_report["iou"], 4))

print("YOLO mask mAP50:",round(instance_report["mask_map50"], 4))
print("YOLO mask mAP50-95:", round(instance_report["mask_map50_95"], 4))

print("Test used for tuning:", False)

BLOCK 6A COMPLETE
U-Net test Dice: 0.7635
U-Net test IoU: 0.6175
YOLO mask mAP50: 0.5464
YOLO mask mAP50-95: 0.1794
Test used for tuning: False


# Compact Deployment Version

In [21]:
VAL_IMAGES_DIR = DATASET_ROOT/ "images" / "val"
VAL_LABELS_DIR = DATASET_ROOT / "labels" / "val"

UNET_ONNX_PATH =  OUTPUT_ROOT / "unet.onnx"
YOLO_ONNX_PATH = OUTPUT_ROOT / "yolo26n_seg.onnx"

UNET_PARITY_PATH = OUTPUT_ROOT / "unet_onnx_parity.json"
YOLO_PARITY_PATH = OUTPUT_ROOT / "yolo_onnx_parity.json"

LATENCY_PATH = OUTPUT_ROOT / "latency_benchmark.csv"

print("ONNX providers:", ort.get_available_providers())

ONNX providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


### U-Net export and parity

**Export U-Net**

In [22]:
unet_cpu = copy.deepcopy(unet_model).cpu().eval()
dummy_input = torch.randn(1, 3, UNET_IMAGE_SIZE, UNET_IMAGE_SIZE)

torch.onnx.export(
    unet_cpu, dummy_input, str(UNET_ONNX_PATH),
    input_names=["images"], output_names=["logits"],
    opset_version=17, dynamo=False
)

onnx_model = onnx.load(str(UNET_ONNX_PATH))
onnx.checker.check_model(onnx_model)

unet_ort_session = ort.InferenceSession(
    str(UNET_ONNX_PATH), providers=["CPUExecutionProvider"]
)

unet_input_names = unet_ort_session.get_inputs()[0].name

print("U-Net ONNX valid:", UNET_ONNX_PATH)

/tmp/ipykernel_3208/4155104011.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


U-Net ONNX valid: /content/drive/MyDrive/vision_unit_02_outputs/block_06/unet.onnx


In [ ]:
validation_dataset = (
    CrackSemanticTestDataset(
        VAL_IMAGES_DIR,
        VAL_LABELS_DIR,
        UNET_IMAGE_SIZE,
    )
)

PARITY_SAMPLE_COUNT = 12

parity_indices = np.linspace(
    0,
    len(validation_dataset) - 1,
    PARITY_SAMPLE_COUNT,
    dtype=int,
)

print(
    "Validation images:",
    len(validation_dataset),
)

print(
    "Parity samples:",
    len(parity_indices),
)